In [1]:

import torch
import numpy as np
from tqdm.autonotebook import tqdm
from matplotlib import pyplot as plt
import torch.nn.functional as F
import torch.nn as nn
import warnings
import copy
from collections import deque
from torch.distributions import Categorical
import pandas as pd

try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False

from model import *
from utils import *
from env import *
from ppo import *

log_wandb = True

import datetime


now = datetime.datetime.now()   
curr_time = now.strftime("%H_%M_%S")
curr_date = now.strftime("%d_%m_%Y")
curr_date_time = str('d_')+curr_date+str('_t_')+curr_time

PROJECT_NAME = "Bio Env"

/var/folders/9v/hs9qlpdx4791kpqgpt09stch0000gn/T/ipykernel_12882/3969684013.py:3: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [ ]:
LAMBDA_ARR = np.array([0.2, 0.4, 0.6 , 0.8, 0.95])
GAMMA_ARR = np.array([0.2, 0.4, 0.6 , 0.8, 0.99])
DELTA_ARR = np.array([0.2, 0.4, 0.6, 0.8, 1.0]) #np.array([0.2, 0.4, 0.6 , 0.8, 1.0])

In [ ]:
# Food env args
FOOD_FOLDER = "foods_dataset"       # directory containing nutrient CSVs
MENU_SIZE   = 2 #2
MAX_STEPS   = 480 #720 #100 # 480 ~ 16 hrs
SEED        = 0 #2000

ENV_ARGS = dict(num_foods=MENU_SIZE,
            max_steps=MAX_STEPS,
            one_hot_embedding=True,
            embed_size=None,
            seed=SEED,)



# ── Environment ───────────────────────────────────────────────────────────
# All nutrient-specific config (decay, window, target, reward weight) is
# read from NUTRIENT_CONFIG in bio_env.py automatically.
env = FoodEnv(
    food_folder=FOOD_FOLDER,
    **ENV_ARGS
)


print("\n── Environment summary ──────────────────────────────────────")
print(f"  Food items      : {env.num_items}")
print(f"  Nutrients       : {env.num_nutrients}  {env.nutrient_names}")
print(f"  Menu size       : {env.num_foods}  (+1 skip → {env.action_space.n} actions)")
print(f"  Max steps       : {env.max_steps}")
print(f"  Target (normed) : {env._norm_targets}")
print()
print(env.nutrient_norm_summary().to_string(index=False))
print("─────────────────────────────────────────────────────────────\n")

[FoodEnv] Loading  'glucose'  from  '/Users/charithapalika/Desktop/Lab projects/Appetite modelling/Single_agent_model/BioEnvWork/foods_dataset/serum_glucose.csv'
           min=0.0000  max=153.4313   foods=28   time_points=500
[FoodEnv] Loading  'peptides'  from  '/Users/charithapalika/Desktop/Lab projects/Appetite modelling/Single_agent_model/BioEnvWork/foods_dataset/small_peptides_absorbed.csv'
           min=0.0000  max=0.0018   foods=28   time_points=500
[FoodEnv] Loading  'fatty_acids'  from  '/Users/charithapalika/Desktop/Lab projects/Appetite modelling/Single_agent_model/BioEnvWork/foods_dataset/fatty_acids_absorbed.csv'
           min=0.0000  max=0.0049   foods=28   time_points=500
[FoodEnv] Ready — 28 foods | 3 time-series nutrients | 0 cumulative nutrients

── Environment summary ──────────────────────────────────────
  Food items      : 28
  Nutrients       : 3  ['glucose', 'peptides', 'fatty_acids']
  Menu size       : 2  (+1 skip → 3 actions)
  Max steps       : 480
  Targ

In [4]:
for lam in LAMBDA_ARR:
    for gamma in GAMMA_ARR:
        for delta_lim in DELTA_ARR:
            AGENT_ARGS = dict(
                            gamma=gamma,
                            lam=lam,
                            limit_delta = delta_lim,
                            clip_eps=0.2,
                            value_coeff=0.5,
                            entropy_coeff=0.01,
                            max_grad_norm=0.5,
                            shared=False,
                            seed=SEED,
                            
                                    )
            
            TRAINING_ARGS = dict(
                            num_episodes=100,#750,
                            actor_lr=1e-4,
                            critic_lr=1e-2,
                            rollout_steps=128,
                            ppo_epochs=4,
                            minibatch_size=32,
                            log_every_episodes=1,
                            rolling_window=50,
                            )
            
            RUN_NAME = "LAM_" + str(lam) + "_GAM_" + str(gamma) + "_DEL_" + str(delta_lim) 
        
            wandb.init(
            project=PROJECT_NAME,
            name=RUN_NAME,
            config={
                "agent_args": AGENT_ARGS,
                "training_params": TRAINING_ARGS,
                "env_args": ENV_ARGS,
                "datetime": curr_date_time,
            }
            )

            agent = PPOAgent(env, device="cpu", **AGENT_ARGS)

            returns, eaten = agent.train(
                                    log_wandb= log_wandb,
                                    printing = False,
                                    **TRAINING_ARGS 
                                    )
            memory, episode_df = agent.generate_episode(log_wandb=log_wandb)

 

            


wandb: Currently logged in as: charithapalika (charitha_palika) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


train/actor_loss,▄▂▁▂▃▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇█████████████████
train/critic_loss,█▆▅▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/entropy,███▇▆▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/foods_eaten,█▅▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/return,▁███████████████████████████████████████
train/total_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/actor_loss,-0.00398
train/critic_loss,30292456.18685
train/entropy,0.12994
train/foods_eaten,7
train/return,255.77225
